<a href="https://colab.research.google.com/github/zydanne-costa/Ondas_ADCP_SCO_Mar_Nov_2025/blob/main/main_strk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

O objetivo principal dessa rotina é extrair altura e período de onda dos arquivos de surfacetrack.

Após o cálculos os df são salvos no diretório '/content/drive/MyDrive/Ondas/Dados/Refinados/'

# 1.IMPORTAÇÃO DE BIBLIOTECAS

In [3]:
import pandas as pd # manipulacao de tempo e dataframes
import numpy as np # calculos
import os # caminhos de diretorios
from glob import glob #trabalhar com arquivos/ listas
import matplotlib.pyplot as plt # graficos
import matplotlib.ticker as mticker
import matplotlib.dates as mdates

Esta célula de código importa todas as bibliotecas Python necessárias para manipulação de dados, cálculos numéricos, operações de sistema de arquivos, plotagem e formatação de datas.

# 2.ACESSO AO DRIVE

In [4]:
from google.colab import drive # import do modulo drive de google.colab
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 3.FUNÇÕES AUXILIARES

## WINDOW

Função para janela de suavização

In [5]:
def window(N, wt):

    nn = N - 1
    pn = 2 * np.pi * np.arange(0, nn + 1) / nn

    if wt[:4] == 'rect': w = np.ones(N)

    elif wt[:4] == 'blac': # blackman Window (filtro)
        w = (0.42- 0.5 * np.cos(pn)+ 0.08 * np.cos(2 * pn))

    else:
        raise ValueError('Tipo de janela inválido')

    return w # w = vetor de pesos

    # redução dos ruídos, efeitos de borda e suavização

Esta função `window(N, wt)` gera um array de janela (pesos) para processamento de sinal. Ela suporta uma janela retangular (`'rect'`) ou uma janela Blackman (`'blac'`). A janela Blackman é tipicamente usada para redução de ruído, efeitos de borda e suavização na análise de sinais.

##WEIM
Aplicar média movel ponderada após percorrer a série de dados (x) utilizando a janela Blackman.

Estimo que a maré tem uma tendencia lenta, com isso consigo 'capturar' o sinal da maré

In [6]:
def weim(N, wt, x):

    w = window(N, wt)

    x = np.array(x).flatten()

    ln = (N - 1) // 2
    lx = len(x)
    lf = lx - ln

    y = np.zeros_like(x)

    for i in range(lx):

        # início
        if i < ln:

            y[i] = (
                np.sum(x[:ln+i+1] * w[ln-i:])
                / np.sum(w[ln-i:])
            )

        # centro
        elif i >= ln and i < lf:

            y[i] = (
                np.sum(x[i-ln:i+ln+1] * w)
                / np.sum(w)
            )

        # final
        else:

            y[i] = (np.sum(x[i-ln:] * w[:len(x[i-ln:])])/ np.sum(w[:len(x[i-ln:])]))

    return y

A função `weim(N, wt, x)` aplica uma média móvel ponderada à série de entrada `x` usando uma janela gerada pela função `window`. Ela lida com casos de borda para o início e o fim da série, suavizando efetivamente os dados para extrair um sinal de 'maré' dos dados amostrados.

# 4.FUNÇÕES DE PROCESSAMENTO

## processar_strk

Aqui eu leio o txt bruto, concateno em um df com tempo e altura de onda

In [7]:
#STRK

def processar_strk(
    caminho_pasta,
    nome_saida
):
    # leitura dos arquivos .txt
    arquivos = sorted(
        glob(
            os.path.join(
                caminho_pasta,
                '*.txt'
            )
        )
    )

    lista_df = []

    for arq in arquivos:

        nome_arquivo = os.path.basename(arq)

        # =================================================
        # DATA
        # =================================================

        data_str = (
            nome_arquivo # extraio a data a partir do nome com AA/MM/DD/HH:MM:ss:mm
            .replace('STrk', '')
            .replace('.txt', '')
        )

        data_inicial = pd.to_datetime(
            data_str,
            format='%Y%m%d%H%M%S%f' # isso define o inicio do arquivo
        )

        # =================================================
        # LEITURA
        # =================================================

        dados = np.loadtxt(
            arq,
            comments='%'
        ) # lendo os arquivos e separando por coluna

        # garantir matriz Nx4
        if dados.ndim == 1:

            continue

        # =================================================
        # INVÁLIDOS
        # =================================================

        dados[dados == 0] = np.nan

        dados[dados == -32768] = np.nan

        # =================================================
        # mm -> m
        # =================================================

        dados = dados / 1000

        # =================================================
        # DATAFRAME
        # =================================================

        dados = pd.DataFrame(
            dados,
            columns=[
                'B1_m',
                'B2_m',
                'B3_m',
                'B4_m'
            ]
        )

        # =================================================
        # INTERPOLAÇÃO
        # apenas gaps pequenos
        # =================================================

        for col in [
            'B1_m',
            'B2_m',
            'B3_m',
            'B4_m'
        ]:

            dados[col] = (
                dados[col]
                .interpolate(
                    limit=4 # interpola até 2 segundos de falha
                )
            )

        # =================================================
        # REMOVER MARÉ
        # =================================================

        dados['B1_h'] = ( # n amostral
            dados['B1_m']
            - weim( # sinal de maré sendo subtraidos do n amostral
                275,
                'blac',
                dados['B1_m'] # o resultado é o sinal de ondas (n_wave = n_amostral - n_maré)
            )
        )

        dados['B2_h'] = (
            dados['B2_m']
            - weim(
                275,
                'blac',
                dados['B2_m']
            )
        )

        dados['B3_h'] = (
            dados['B3_m']
            - weim(
                275,
                'blac',
                dados['B3_m']
            )
        )

        dados['B4_h'] = (
            dados['B4_m']
            - weim(
                275,
                'blac',
                dados['B4_m']
            )
        )

        # =================================================
        # TEMPO ORIGINAL
        # 2 Hz = 0.5 s
        # =================================================

        tempo = pd.date_range(
            start=data_inicial,
            periods=len(dados),
            freq='500ms'
        )

        dados['DataHora'] = tempo

        # =================================================
        # DATAFRAME FINAL
        # =================================================

        dados_final = dados[
            [
                'DataHora',
                'B1_h',
                'B2_h',
                'B3_h',
                'B4_h'
            ]
        ]

        lista_df.append(
            dados_final
        )

    # =====================================================
    # CONCATENAR
    # =====================================================

    if len(lista_df) == 0:
      raise ValueError('Nenhum arquivo encontrado')

    df_final = pd.concat(
        lista_df,
        ignore_index=True
    )

    # ordenar
    df_final = df_final.sort_values(
        'DataHora'
    )

    # reset índice
    df_final = df_final.reset_index(
        drop=True
    )

    # =====================================================
    # CORTES TEMPORAIS
    # =====================================================

    if nome_saida == 'Strk_CHU': # chuvoso

        inicio = pd.to_datetime(
            '2025-03-27 13:20:00'
        )

        fim = pd.to_datetime(
            '2025-04-02 16:00:00' # 6 dias 2 h e 40 min / 146,66 horas
        )

        df_final = df_final[
            (
                df_final['DataHora']
                >= inicio
            )
            &
            (
                df_final['DataHora']
                <= fim
            )
        ]

    if nome_saida == 'Strk_SEC':

        inicio = pd.to_datetime(
            '2025-11-22 18:00:00'
        )

        fim = pd.to_datetime(
            '2025-11-30 09:00:00' # 7 dias e 15 horas
        )

        df_final = df_final[
            (
                df_final['DataHora']
                >= inicio
            )
            &
            (
                df_final['DataHora']
                <= fim
            )
        ]

        # =================================================
        # PADRONIZAR DURAÇÃO TEMPORAL
        # =================================================

        duracao_chu = (

            pd.to_datetime(
                '2025-04-02 16:00:00'
            )

            -

            pd.to_datetime(
                '2025-03-27 13:20:00'
            )

        )

        novo_fim = (
            inicio
            + duracao_chu
        )

        df_final = df_final[
            df_final['DataHora']
            <= novo_fim
        ]

    # =====================================================
    # RESET ÍNDICE
    # =====================================================

    df_final = df_final.reset_index(
        drop=True
    )

    # =====================================================
    # SALVAR CSV
    # =====================================================

    caminho_saida = (
        '/content/drive/MyDrive/'
        'Ondas/Dados/Refinados/'
        f'{nome_saida}.csv'
    )

    df_final.to_csv(
        caminho_saida,
        index=False,
        date_format='%Y-%m-%d %H:%M:%S.%f'
    )

    return df_final

A função `processar_strk` foi projetada para ler e processar arquivos surfacetrack `.txt`. Ela executa as seguintes etapas:
1.  **Leitura de Arquivos**: Lê todos os arquivos `.txt` de um diretório especifico.
2.  **Extração de Data**: Extrai a data e hora de início do nome do arquivo.
3.  **Carregamento de Dados**: Carrega dados numéricos dos arquivos `.txt`.
4.  **Tratamento de Dados Inválidos**: Substitui os valores `0` e `-32768` por `NaN` (Não é um Número).
5.  **Conversão de Unidades**: Converte os dados de milímetros para metros.
6.  **Criação de DataFrame**: Organiza os dados em um DataFrame Pandas com as colunas 'B1_m', 'B2_m', 'B3_m', 'B4_m'.
7.  **Interpolação**: Preenche pequenas lacunas (até 2 segundos) nos dados usando interpolação linear.
8.  **Remoção de Maré**: Aplica a função `weim` (com uma janela Blackman) para remover o sinal de maré de cada feixe, resultando em dados de altura de onda ('B1_h', 'B2_h', 'B3_h', 'B4_h').
9.  **Criação de Série Temporal**: Cria uma coluna `DataHora` (DateTime) com base na hora de início extraída e uma frequência de amostragem de 2 Hz (0.5s).
10. **Concatenação de DataFrames**: Combina todos os dataframes processados em um único DataFrame.
11. **Recorte Temporal**: Filtra o DataFrame com base em intervalos de data específicos ('Chuvoso' e 'Seco') e padroniza a duração.
12. **Exportação para CSV**: Salva o DataFrame final processado em um arquivo CSV no diretório 'Refinados' especificado.

# 5.FUNÇÃO DE ONDAS

Utilizando zero-upcrossing, cruzamento ascendente do nível zero

## individual_waves

In [8]:
# =========================================================
# ONDAS INDIVIDUAIS
# MÉTODO ZERO-UPCROSSING
# =========================================================

def individual_waves(
    serie,
    tempo,
):

    # =====================================================
    # ARRAY
    # =====================================================

    serie = np.array(serie)

    tempo = pd.Series(
        pd.to_datetime(tempo)
    ).reset_index(drop=True)

    # =====================================================
    # REMOVER NaN
    # =====================================================

    mascara = ~np.isnan(serie)

    serie = serie[mascara]

    tempo = tempo[mascara]

    # =====================================================
    # REMOVER NÍVEL MÉDIO
    # =====================================================

    serie = serie - np.mean(serie)

    # =====================================================
    # CRUZAMENTOS ASCENDENTES
    # =====================================================

    cruzasc = np.where(

        (serie[:-1] < 0)

        &

        (serie[1:] >= 0)

    )[0]

    # =====================================================
    # LISTAS
    # =====================================================

    alturas = []

    periodos = []

    tempo_ondas = []

    # =====================================================
    # LOOP DAS ONDAS
    # =====================================================

    for i in range(len(cruzasc)-1):

        ini = cruzasc[i]

        fim = cruzasc[i+1]

        trecho = serie[
            ini:fim
        ]

        if len(trecho) > 0:

            # =============================================
            # ALTURA
            # =============================================

            H = (
                np.max(trecho)
                -
                np.min(trecho)
            )

            # =============================================
            # PERÍODO
            # =============================================

            T = (
                tempo.iloc[fim]
                -
                tempo.iloc[ini]
            ).total_seconds()

            # filtro fisico para tempo
            # filtro físico
            if T > 30:
                continue

            # =============================================
            # TEMPO CENTRAL DA ONDA
            # =============================================

            tempo_central = (
                tempo.iloc[ini]
                +
                (
                    tempo.iloc[fim]
                    -
                    tempo.iloc[ini]
                ) / 2
            )

            alturas.append(H)

            periodos.append(T)

            tempo_ondas.append(
                tempo_central
            )

    # =====================================================
    # DATAFRAME FINAL
    # =====================================================

    df_ondas = pd.DataFrame({

        'DataHora': tempo_ondas,

        'H': alturas,

        'T': periodos

    })

    return df_ondas

A função `individual_waves(serie, tempo)` identifica ondas individuais e calcula sua altura (H) e período (T) usando o método de cruzamento de zero ascendente. Ela executa as seguintes etapas:
1.  **Preparação dos Dados**: Converte a série de entrada e o tempo para arrays NumPy e Series Pandas, respectivamente.
2.  **Remoção de NaN**: Remove valores ausentes (`NaN`) da série e dos pontos de tempo correspondentes.
3.  **Remoção do Nível Médio**: Subtrai a média da série para centralizá-la em torno de zero.
4.  **Detecção de Cruzamento de Zero Ascendente**: Identifica os pontos onde a série cruza zero de negativo para positivo.
5.  **Cálculo dos Parâmetros da Onda**: Para cada onda identificada (entre dois cruzamentos de zero ascendentes consecutivos):
    *   **Altura (H)**: Calculada como a diferença entre os valores máximo e mínimo dentro do segmento da onda.
    *   **Período (T)**: Calculado como a diferença de tempo entre os dois cruzamentos de zero ascendentes. Um filtro físico é aplicado para excluir períodos maiores que 30 segundos.
    *   **Tempo Central**: Determina o ponto de tempo central de cada onda.
6.  **Saída do DataFrame**: Retorna um DataFrame contendo `DataHora` (tempo central), `H` (altura) e `T` (período) para cada onda individual.

## H_T

In [9]:
# =========================================================
# EXTRAIR H E T DE TODOS OS BEAMS
# =========================================================

def processar_ondas_individuais(df):

    resultados = []

    beams = [
        'B1_h',
        'B2_h',
        'B3_h',
        'B4_h'
    ]

    for beam in beams:

        print(f'Processando {beam}...')

        df_ondas = individual_waves(

            serie=df[beam],

            tempo=df['DataHora']

        )

        df_ondas = df_ondas.rename(
            columns={

                'H': f'{beam[:2]}_H',

                'T': f'{beam[:2]}_T'

            }
        )

        resultados.append(df_ondas)

    # =====================================================
    # CONCATENAR
    # =====================================================

    df_final = pd.concat(
        resultados,
        axis=1
    )

    # remover colunas repetidas DataHora
    df_final = df_final.loc[
        :,
        ~df_final.columns.duplicated()
    ]

    return df_final

A função `processar_ondas_individuais(df)` aplica a função `individual_waves` a cada um dos quatro feixes (`B1_h`, `B2_h`, `B3_h`, `B4_h`) no DataFrame de entrada. Em seguida, concatena os resultados, renomeando as colunas de altura e período para incluir o identificador do feixe (por exemplo, `B1_H`, `B1_T`). Finalmente, remove as colunas `DataHora` duplicadas resultantes da concatenação, fornecendo um único DataFrame com parâmetros de onda individuais para todos os feixes.

## calc_hs_beams
encontro ondas individuais em cada beam, junto todas, coloco em ordem decrescente, pego 1/3 das maiores e faço a média.

In [10]:
def calc_hs_beams(df):

    alturas_total = []

    beams = [
        'B1_h',
        'B2_h',
        'B3_h',
        'B4_h'
    ]

    for beam in beams:

        df_ondas = individual_waves(
            serie=df[beam],
            tempo=df['DataHora']
        )

        alturas_total.extend(
            df_ondas['H'].values
        )

    alturas_total = np.array(
        alturas_total
    )

    if len(alturas_total) == 0:

        return np.nan

    # ordenar decrescente
    alturas_total = np.sort(
        alturas_total
    )[::-1]

    # 1/3 maiores
    n = int(
        len(alturas_total) / 3
    )

    if n == 0:

        return np.nan

    hs = np.mean(
        alturas_total[:n]
    )

    return hs

A função `calc_hs_beams(df)` calcula a **altura significativa da onda (Hs)** para todos os feixes combinados. Ela funciona da seguinte forma:
1.  **Alturas de Ondas Individuais**: Itera por cada feixe ('B1_h' a 'B4_h') e usa `individual_waves` para extrair as alturas de ondas individuais (`H`) para cada feixe.
2.  **Alturas Agregadas**: Todas as alturas de ondas individuais de todos os feixes são coletadas em um único array `alturas_total`.
3.  **Ordenar e Selecionar**: O array `alturas_total` é ordenado em ordem decrescente, e o terço superior das maiores alturas de ondas é selecionado.
4.  **Calcular Hs**: A altura significativa da onda (Hs) é calculada como a média dessas alturas de ondas do terço superior. Se nenhuma onda for encontrada, ela retorna `np.nan`.

## calc_hs_intervalado

In [11]:
# =========================================================
# RESUMO DE PARÂMETROS DE ONDA POR INTERVALO
# =========================================================

def calc_resumo_intervalo(df, intervalo):

    resultados = []

    grupos = df.resample(
        intervalo,
        on='DataHora'
    )

    for tempo, grupo in grupos:

        alturas_total = []
        periodos_total = []

        # =============================================
        # CALCULAR ONDAS EM CADA BEAM
        # =============================================

        for beam in [
            'B1_h',
            'B2_h',
            'B3_h',
            'B4_h'
        ]:

            dados_validos = grupo[
                ['DataHora', beam]
            ].dropna()

            if len(dados_validos) > 10:

                df_ondas = individual_waves(
                    serie=dados_validos[beam],
                    tempo=dados_validos['DataHora']
                )

                alturas_total.extend(
                    df_ondas['H'].values
                )

                periodos_total.extend(
                    df_ondas['T'].values
                )

        # =============================================
        # CALCULAR PARÂMETROS
        # =============================================

        alturas_total = np.array(alturas_total)
        periodos_total = np.array(periodos_total)

        if len(alturas_total) == 0:
            continue

        ordem = np.argsort(
            alturas_total
        )[::-1]

        alturas_ord = alturas_total[ordem]
        periodos_ord = periodos_total[ordem]

        n = int(len(alturas_ord)/3)

        if n == 0:
            continue

        Hs = np.mean(
            alturas_ord[:n]
        )

        Ts = np.mean(
            periodos_ord[:n]
        )

        Hmax = np.max(
            alturas_total
        )

        Tmax = periodos_total[
            np.argmax(alturas_total)
        ]

        Hmed = np.mean(
            alturas_total
        )

        Tmed = np.mean(
            periodos_total
        )

        Nondas = len(
            alturas_total
        )

        resultados.append([
            tempo,
            Hs,
            Ts,
            Hmax,
            Tmax,
            Hmed,
            Tmed,
            Nondas
        ])

    df_resumo = pd.DataFrame(

        resultados,

        columns=[

            'DataHora',

            'Hs',

            'Ts',

            'Hmax',

            'Tmax',

            'Hmed',

            'Tmed',

            'Nondas'

        ]

    )

    if len(df_resumo) > 0:

        df_resumo['Hs'] = df_resumo['Hs'].round(2)
        df_resumo['Ts'] = df_resumo['Ts'].round(2)

        df_resumo['Hmax'] = df_resumo['Hmax'].round(2)
        df_resumo['Tmax'] = df_resumo['Tmax'].round(2)

        df_resumo['Hmed'] = df_resumo['Hmed'].round(2)
        df_resumo['Tmed'] = df_resumo['Tmed'].round(2)

    return df_resumo

# 6.PROCESSAMENTO DOS **DADOS**

## Chuvoso

In [12]:
# CHUVOSO

caminho_chu = (
    '/content/drive/MyDrive/'
    'Ondas/Dados/marco2025/'
    'SCO1/Strk'
)

Strk_CHU = processar_strk(
    caminho_chu,
    'Strk_CHU'
)

Esta célula define o caminho do arquivo para os dados do período 'Chuvoso' (estação chuvosa) e então chama a função `processar_strk` para processar esses arquivos, armazenando o DataFrame resultante em `Strk_CHU`.

## Menos Chuvosos

In [13]:
# MENOS CHUVOSO

caminho_sec = (
    '/content/drive/MyDrive/'
    'Ondas/Dados/novembro2025/'
    'SCO2/STrk_2'
)

Strk_SEC = processar_strk(
    caminho_sec,
    'Strk_SEC'
)


Esta célula define o caminho do arquivo para os dados do período 'Menos Chuvosos' (estação menos chuvosa) e então chama a função `processar_strk` para processar esses arquivos, armazenando o DataFrame resultante em `Strk_SEC`.

## DF Individuais

In [14]:
# com parametros
df_ondas_CHU = processar_ondas_individuais(
    Strk_CHU
)

df_ondas_SEC = processar_ondas_individuais(
    Strk_SEC
)

Processando B1_h...
Processando B2_h...
Processando B3_h...
Processando B4_h...
Processando B1_h...
Processando B2_h...
Processando B3_h...
Processando B4_h...


Estas células aplicam a função `processar_ondas_individuais` aos dataframes `Strk_CHU` e `Strk_SEC` para extrair os parâmetros de ondas individuais (altura e período para cada feixe) e armazená-los em `df_ondas_CHU` e `df_ondas_SEC`, respectivamente.

In [15]:
df_strk_chu = Strk_CHU.copy()

df_strk_sec = Strk_SEC.copy()

# datetime

df_strk_chu['DataHora'] = pd.to_datetime(
    df_strk_chu['DataHora']
)

df_strk_sec['DataHora'] = pd.to_datetime(
    df_strk_sec['DataHora']
)


Estas células criam cópias dos dataframes `Strk_CHU` e `Strk_SEC` como `df_strk_chu` e `df_strk_sec`. Elas também convertem explicitamente a coluna 'DataHora' em ambos os novos dataframes para objetos datetime, garantindo operações de série temporal adequadas nas etapas subsequentes.

# 7.Formatação dos plots

In [16]:
plt.rcParams['font.family'] = 'serif'

# vírgula decimal

def comma_formatter(x, pos):

    return (
        f'{x:,.2f}'
        .replace('.', 'X')
        .replace(',', '.')
        .replace('X', ',')
    )

Esta célula define a família de fontes padrão para os gráficos do Matplotlib como 'serif' e define uma função `comma_formatter`. Este formatador personalizado é usado para formatar números com vírgulas decimais (comum em algumas localidades) para melhor legibilidade nos eixos dos gráficos.

# 8.CÁLCULO DE Hs

## Geral

In [17]:
HsChu_strk = calc_hs_beams(
    df_strk_chu
)

HsSec_strk = calc_hs_beams(
    df_strk_sec
)

print(
    f'Hs STRK Chuvoso = '
    f'{HsChu_strk:.3f} m'
)

print(
    f'Hs STRK Seco = '
    f'{HsSec_strk:.3f} m'
)

Hs STRK Chuvoso = 0.772 m
Hs STRK Seco = 0.749 m


Esta célula calcula a altura significativa da onda (Hs) geral para os períodos 'Chuvoso' (`df_strk_chu`) e 'Seco' (`df_strk_sec`) usando a função `calc_hs_beams`. Em seguida, imprime os valores de Hs calculados, formatados com três casas decimais.

## Intervalado

Estas células calculam a altura significativa da onda (Hs) para os períodos 'Chuvoso' e 'Seco', desta vez em intervalos de 30 minutos, usando a função `calc_hs_intervalo`. Os resultados são armazenados em `df_Hs_30min_CHU_STRK` e `df_Hs_30min_SEC_STRK`.

### Hs 1H

In [18]:
df_Resumo_1H_CHU = calc_resumo_intervalo(
    df_strk_chu,
    '1H'
)


df_Resumo_1H_SEC = calc_resumo_intervalo(
    df_strk_sec,
    '1H'
)

/tmp/ipykernel_7065/3911290890.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  grupos = df.resample(
/tmp/ipykernel_7065/3911290890.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  grupos = df.resample(


Estas células calculam a altura significativa da onda (Hs) para os períodos 'Chuvoso' e 'Seco', desta vez em intervalos de 1 hora, usando a função `calc_hs_intervalo`. Os resultados são armazenados em `df_Hs_1H_CHU_STRK` e `df_Hs_1H_SEC_STRK`.

In [19]:
df_Resumo_1H_CHU.head()

,DataHora,Hs,Ts,Hmax,Tmax,Hmed,Tmed,Nondas
0,2025-03-27 13:00:00,0.42,4.64,0.75,5.0,0.14,3.81,293
1,2025-03-27 14:00:00,0.54,4.25,1.00,10.0,0.45,4.13,584
2,2025-03-27 15:00:00,0.84,3.71,1.27,5.0,0.52,2.84,2153
3,2025-03-27 16:00:00,0.57,3.40,1.00,9.0,0.40,2.92,2086
4,2025-03-27 17:00:00,0.65,3.46,1.30,2.5,0.41,2.83,2157


In [22]:
df_Resumo_1H_SEC.head()

,DataHora,Hs,Ts,Hmax,Tmax,Hmed,Tmed,Nondas
0,2025-11-22 23:00:00,0.52,3.70,0.77,3.0,0.37,2.85,712
1,2025-11-23 00:00:00,0.94,4.32,1.00,6.5,0.69,3.11,655
2,2025-11-23 01:00:00,1.12,4.23,1.57,6.0,0.76,3.36,603
3,2025-11-23 02:00:00,0.50,4.17,0.54,13.5,0.46,3.87,489
4,2025-11-23 03:00:00,0.94,6.97,1.00,4.5,0.81,7.62,233


# 9.EXPORTAÇÃO

## Exportação p/ 'Refinados"

In [20]:
pasta_saida = ('/content/drive/MyDrive/'
    'Ondas/Dados/Refinados/')

df_ondas_CHU.to_csv(pasta_saida + 'Ondas_CHU.csv',index=False)

df_ondas_SEC.to_csv(pasta_saida + 'Ondas_SEC.csv',index=False)

Estas células exportam os DataFrames `df_ondas_CHU` e `df_ondas_SEC` (contendo parâmetros de ondas individuais) para arquivos CSV nomeados `Ondas_CHU.csv` e `Ondas_SEC.csv` respectivamente, dentro da pasta de saída especificada. `index=False` impede a gravação do índice do DataFrame no CSV.

In [23]:
pasta_saida = ('/content/drive/MyDrive/Ondas/Dados/Refinados/')

# Hs 1H
df_Resumo_1H_CHU.to_csv(pasta_saida + 'WaveData_1H_CHU.txt',index=False, sep='\t')
df_Resumo_1H_SEC.to_csv(pasta_saida + 'WaveData_1H_SEC.txt',index=False, sep='\t')

# Hs 3H
#df_Hs_3H_CHU_STRK.to_csv(pasta_saida + 'Hs_3H_CHU.txt',index=False, sep='\t')
#df_Hs_3H_SEC_STRK.to_csv(pasta_saida + 'Hs_3H_SEC.txt',index=False, sep='\t')